In [1]:
# =====================================================
# LOAD MASTER CUSTOMER DATASET
# Purpose: Import pandas and load the cleaned master
# customer dataset for Python data-quality analysis
# =====================================================

import pandas as pd

df = pd.read_excel("master_customers_sql.xlsx")

print("Dataset loaded successfully.")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Dataset loaded successfully.
Number of rows: 601
Number of columns: 17


In [2]:
# =====================================================
# INSPECT DATASET STRUCTURE
# Purpose: Review column names, data types, non-null
# counts, and overall structure of the master dataset
# =====================================================

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 601 entries, 0 to 600
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Customer_ID                  601 non-null    object        
 1   Full_Name                    601 non-null    object        
 2   Phone                        585 non-null    object        
 3   Email                        592 non-null    object        
 4   State                        601 non-null    object        
 5   Customer_Type                601 non-null    object        
 6   Registration_Date            594 non-null    object        
 7   Source_Channel               601 non-null    object        
 8   Branch                       601 non-null    object        
 9   Record_Load_Date             601 non-null    datetime64[ns]
 10  Data_Quality_Status          601 non-null    object        
 11  Customer_ID_Counts.ID_Count  601 non-null    

In [3]:
# =====================================================
# CUSTOMER ID UNIQUENESS CHECK
# Purpose: Confirm that each customer appears only once
# in the final master dataset
# =====================================================

duplicate_ids = df["Customer_ID"].duplicated().sum()
unique_ids = df["Customer_ID"].nunique()

print("Total records:", len(df))
print("Unique Customer IDs:", unique_ids)
print("Duplicate Customer IDs:", duplicate_ids)

Total records: 601
Unique Customer IDs: 601
Duplicate Customer IDs: 0


In [4]:
# =====================================================
# MISSING VALUE PROFILE
# Purpose: Identify missing values across all columns
# and calculate their percentage of the dataset
# =====================================================

missing_profile = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_profile = missing_profile[missing_profile["Missing_Count"] > 0]

print(missing_profile)

                         Missing_Count  Missing_Percentage
Phone                               16                2.66
Email                                9                1.50
Registration_Date                    7                1.16
Registration_Date_Clean              7                1.16


In [5]:
# =====================================================
# INDEPENDENT DATA QUALITY VALIDATION
# Purpose: Recalculate valid and invalid records using
# the core customer data-quality rules
# =====================================================

def check_quality(row):
    if pd.isna(row["Phone"]) or str(row["Phone"]).strip() == "":
        return "Invalid"

    if pd.isna(row["Email"]) or str(row["Email"]).strip() == "":
        return "Invalid"

    email = str(row["Email"]).strip()

    if "@" not in email or "." not in email:
        return "Invalid"

    return "Valid"


df["Python_Quality_Status"] = df.apply(check_quality, axis=1)

print(df["Python_Quality_Status"].value_counts())

Python_Quality_Status
Valid      568
Invalid     33
Name: count, dtype: int64


In [6]:
# =====================================================
# CROSS-TOOL QUALITY VALIDATION
# Purpose: Compare Python's quality classification with
# the existing Power Query classification record-by-record
# =====================================================

mismatches = df[
    df["Python_Quality_Status"] != df["Data_Quality_Status"]
]

print("Matching classifications:", len(df) - len(mismatches))
print("Mismatched classifications:", len(mismatches))

Matching classifications: 601
Mismatched classifications: 0


In [7]:
# =====================================================
# DATA QUALITY BY SOURCE CHANNEL
# Purpose: Compare valid and invalid customer records
# across each source system
# =====================================================

source_quality = pd.crosstab(
    df["Source_Channel"],
    df["Python_Quality_Status"]
)

source_quality["Total"] = source_quality.sum(axis=1)

source_quality["Valid_Rate_Percent"] = (
    source_quality["Valid"] / source_quality["Total"] * 100
).round(2)

print(source_quality)

Python_Quality_Status  Invalid  Valid  Total  Valid_Rate_Percent
Source_Channel                                                  
Branch                      20    203    223               91.03
Legacy System                8    199    207               96.14
Website                      5    166    171               97.08


In [8]:
# =====================================================
# CUSTOMER TYPE DISTRIBUTION
# Purpose: Analyse the number and percentage of customers
# across each customer segment
# =====================================================

customer_type_analysis = pd.DataFrame({
    "Customer_Count": df["Customer_Type"].value_counts(),
    "Percentage": (
        df["Customer_Type"].value_counts(normalize=True) * 100
    ).round(2)
})

print(customer_type_analysis)

                Customer_Count  Percentage
Customer_Type                             
Retail                     303       50.42
SME                        151       25.12
Corporate                   97       16.14
Small Business              27        4.49
Individual                  18        3.00
Business                     5        0.83


In [9]:
# =====================================================
# DATA QUALITY BY CUSTOMER TYPE
# Purpose: Compare data-quality performance across
# different customer segments
# =====================================================

type_quality = pd.crosstab(
    df["Customer_Type"],
    df["Python_Quality_Status"]
)

type_quality["Total"] = type_quality.sum(axis=1)

type_quality["Valid_Rate_Percent"] = (
    type_quality["Valid"] / type_quality["Total"] * 100
).round(2)

type_quality = type_quality.sort_values(
    "Valid_Rate_Percent",
    ascending=False
)

print(type_quality)

Python_Quality_Status  Invalid  Valid  Total  Valid_Rate_Percent
Customer_Type                                                   
Business                     0      5      5              100.00
Individual                   0     18     18              100.00
Small Business               0     27     27              100.00
Corporate                    4     93     97               95.88
Retail                      19    284    303               93.73
SME                         10    141    151               93.38


In [10]:
# =====================================================
# INVALID RECORD ROOT-CAUSE ANALYSIS
# Purpose: Identify the main data-quality problems
# affecting invalid master customer records
# =====================================================

invalid_df = df[df["Python_Quality_Status"] == "Invalid"].copy()

missing_phone = invalid_df["Phone"].isna().sum()
missing_email = invalid_df["Email"].isna().sum()

malformed_email = (
    invalid_df["Email"].notna()
    & (
        ~invalid_df["Email"].astype(str).str.contains("@", regex=False)
        | ~invalid_df["Email"].astype(str).str.contains(".", regex=False)
    )
).sum()

print("Missing Phone:", missing_phone)
print("Missing Email:", missing_email)
print("Malformed Email:", malformed_email)

Missing Phone: 16
Missing Email: 9
Malformed Email: 9


In [11]:
# =====================================================
# MULTIPLE DATA-QUALITY ISSUE ANALYSIS
# Purpose: Identify invalid customers affected by more
# than one critical data-quality problem
# =====================================================

invalid_df["Missing_Phone_Issue"] = invalid_df["Phone"].isna().astype(int)

invalid_df["Missing_Email_Issue"] = invalid_df["Email"].isna().astype(int)

invalid_df["Malformed_Email_Issue"] = (
    invalid_df["Email"].notna()
    & (
        ~invalid_df["Email"].astype(str).str.contains("@", regex=False)
        | ~invalid_df["Email"].astype(str).str.contains(".", regex=False)
    )
).astype(int)

invalid_df["Issue_Count"] = (
    invalid_df["Missing_Phone_Issue"]
    + invalid_df["Missing_Email_Issue"]
    + invalid_df["Malformed_Email_Issue"]
)

multiple_issues = invalid_df[
    invalid_df["Issue_Count"] > 1
][["Customer_ID", "Full_Name", "Issue_Count"]]

print(multiple_issues)

    Customer_ID    Full_Name  Issue_Count
366  CUST-00384  Peter Ekong            2


In [12]:
# =====================================================
# DATA QUALITY KPI SUMMARY
# Purpose: Create a consolidated summary of the main
# customer master data-quality metrics
# =====================================================

quality_summary = pd.DataFrame({
    "Metric": [
        "Total Master Customers",
        "Unique Customer IDs",
        "Duplicate Customer IDs",
        "Valid Records",
        "Invalid Records",
        "Missing Phone",
        "Missing Email",
        "Malformed Email"
    ],
    "Value": [
        len(df),
        df["Customer_ID"].nunique(),
        df["Customer_ID"].duplicated().sum(),
        (df["Python_Quality_Status"] == "Valid").sum(),
        (df["Python_Quality_Status"] == "Invalid").sum(),
        df["Phone"].isna().sum(),
        df["Email"].isna().sum(),
        malformed_email
    ]
})

print(quality_summary)

                   Metric  Value
0  Total Master Customers    601
1     Unique Customer IDs    601
2  Duplicate Customer IDs      0
3           Valid Records    568
4         Invalid Records     33
5           Missing Phone     16
6           Missing Email      9
7         Malformed Email      9


In [13]:
# =====================================================
# EXPORT DATA QUALITY KPI SUMMARY
# Purpose: Save the final Python-generated quality
# metrics as a reusable project output
# =====================================================

quality_summary.to_csv(
    "data_quality_kpi_summary.csv",
    index=False
)

print("Data quality KPI summary exported successfully.")

Data quality KPI summary exported successfully.


In [14]:
# =====================================================
# EXPORT DATA QUALITY EXCEPTION REPORT
# Purpose: Create an actionable list of invalid customer
# records and the specific quality issues affecting them
# =====================================================

exception_report = invalid_df[
    [
        "Customer_ID",
        "Full_Name",
        "Phone",
        "Email",
        "State",
        "Source_Channel",
        "Missing_Phone_Issue",
        "Missing_Email_Issue",
        "Malformed_Email_Issue",
        "Issue_Count"
    ]
].copy()

exception_report.to_csv(
    "customer_data_quality_exceptions.csv",
    index=False
)

print("Exception records exported:", len(exception_report))

Exception records exported: 33


In [15]:
# =====================================================
# EXPORT PYTHON-VALIDATED MASTER DATASET
# Purpose: Save the final customer master dataset with
# independent Python data-quality validation included
# =====================================================

df.to_csv(
    "master_customers_python_validated.csv",
    index=False
)

print("Validated master records exported:", len(df))

Validated master records exported: 601


In [16]:
# =====================================================
# FINAL PROJECT RECONCILIATION
# Purpose: Confirm that all master records are accounted
# for and the final dataset passes integrity checks
# =====================================================

total = len(df)
valid = (df["Python_Quality_Status"] == "Valid").sum()
invalid = (df["Python_Quality_Status"] == "Invalid").sum()
unique_ids = df["Customer_ID"].nunique()
duplicate_ids = df["Customer_ID"].duplicated().sum()

print("Total Master Customers:", total)
print("Valid + Invalid:", valid + invalid)
print("Unique Customer IDs:", unique_ids)
print("Duplicate Customer IDs:", duplicate_ids)
print("Reconciliation Passed:", total == valid + invalid == unique_ids)

Total Master Customers: 601
Valid + Invalid: 601
Unique Customer IDs: 601
Duplicate Customer IDs: 0
Reconciliation Passed: True
